Import potrzebnych bibliotek

In [1]:
import pandas as pd
import numpy as np

Wczytanie danych

In [2]:
df = pd.read_csv("auction_results_color_svd.csv")

df.head()

,ARTIST,TECHNIQUE,SIGNATURE,CONDITION,TOTAL DIMENSIONS,YEAR,Colorfulness Score,SVD Entropy,PRICE
0,218,-1.295300,1,2,-0.157723,-1.039766,51.632554,5.453204,150
1,101,-0.122087,2,2,-0.442668,-0.580107,161.631656,6.154763,270
2,274,-0.122087,2,2,0.263423,-0.626073,117.464780,6.908661,360
3,354,3.397553,0,2,-0.827075,-0.488176,164.609302,6.986244,343
4,354,-0.122087,2,2,-0.145178,0.431142,91.023011,5.859255,150


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22110 entries, 0 to 22109
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ARTIST              22110 non-null  int64  
 1   TECHNIQUE           22110 non-null  float64
 2   SIGNATURE           22110 non-null  int64  
 3   CONDITION           22110 non-null  int64  
 4   TOTAL DIMENSIONS    22110 non-null  float64
 5   YEAR                22110 non-null  float64
 6   Colorfulness Score  22110 non-null  float64
 7   SVD Entropy         22110 non-null  float64
 8   PRICE               22110 non-null  int64  
dtypes: float64(5), int64(4)
memory usage: 1.5 MB


Dopasowanie typów zmiennych

In [3]:
zmienne_kategoryczne =  ['ARTIST', 'TECHNIQUE', 'SIGNATURE', 'CONDITION']
bloki_kategoryczne = []
for zmienna in zmienne_kategoryczne:
    kody = df[zmienna].astype('category').cat.codes.to_numpy(dtype=np.int32)
    liczba_klas = np.max(kody) + 1
    one_hot = np.eye(liczba_klas)[kody]
    bloki_kategoryczne.append(one_hot)


X_kat_cale = np.hstack(bloki_kategoryczne)


Dane treningowe i testowe

In [4]:
np.random.seed(42)
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

indeksy_train = df_train.index
indeksy_test = df_test.index

X_kat_train = X_kat_cale[indeksy_train]
X_kat_test = X_kat_cale[indeksy_test]

Normalizacja zmiennych

In [5]:
zmienne_liczbowe = ["TOTAL DIMENSIONS", "YEAR", "Colorfulness Score", "SVD Entropy"]
srednia = df_train[zmienne_liczbowe].mean()
odchylenie = df_train[zmienne_liczbowe].std()

df_train[zmienne_liczbowe] = (df_train[zmienne_liczbowe] - srednia) / odchylenie
df_test[zmienne_liczbowe] = (df_test[zmienne_liczbowe] - srednia) / odchylenie

Przejście na tablice NumPy

In [7]:
X_num_train = df_train[zmienne_liczbowe].to_numpy(dtype=np.float32)
X_num_test = df_test[zmienne_liczbowe].to_numpy(dtype=np.float32)

X_train = np.hstack([X_kat_train, X_num_train], dtype=np.float32)
X_test = np.hstack([X_kat_test, X_num_test], dtype=np.float32)

srednia_cena = df_train['PRICE'].mean()
odchylenie_cena = df_train['PRICE'].std()

y_train = ((df_train['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
y_test = ((df_test['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)


Tworzenie Dense Layer